# 从零实现 Unigram：lattice、Viterbi、EM 与采样

## 学习目标

完成本 notebook 后，你应该能够：

1. 把字符串的所有合法子词切分表示成 lattice；
2. 用 Viterbi 求最大概率路径，并说明它为何不是最长匹配；
3. 用 log-sum-exp 和 forward-backward 计算边缘概率与期望计数；
4. 解释候选初始化、EM、删除损失与迭代剪枝；
5. 区分 n-best、随机 subword regularization 和确定性推理。

> 代码只依赖 Python 标准库，使用很小的手工词表展示数学结构；生产训练器还需高效 Trie、候选生成和大规模并行统计。


## 1. 原理与公式：概率模型与 lattice

对一条切分 $z=(t_1,\ldots,t_k)$，Unigram 假设：

\[
P(z)=\prod_i p(t_i),\qquad \operatorname{cost}(z)=-\sum_i\log p(t_i).
\]

字符串 $x$ 的观测概率需要对所有合法切分求和：

\[
P(x)=\sum_{z\in S(x)}\prod_{t\in z}p(t).
\]

lattice 的节点是字符串位置；若 `text[i:j]` 是词表 piece，就建立一条 $i\to j$ 的边，边权为 $-\log p(t)$。任意从 0 到末尾的路径就是一种完整切分。


In [ ]:
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
from collections import Counter  # 导入本单元所需的依赖。

# 权重归一化为合法的 unigram 概率。字符项保证示例字符串可达。
weights = {  # 计算并保存当前步骤的中间状态。
    "a": 40.0, "b": 20.0, "c": 25.0,  # 执行当前语句以推进本节示例。
    "ab": 5.0, "bc": 30.0, "abc": 0.5,  # 执行当前语句以推进本节示例。
    "u": 5.0, "n": 5.0, "i": 20.0, "g": 10.0,  # 执行当前语句以推进本节示例。
    "r": 10.0, "m": 10.0, "un": 20.0,  # 执行当前语句以推进本节示例。
    "uni": 50.0, "gram": 50.0,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
normalizer = sum(weights.values())  # 计算并保存当前步骤的中间状态。
probabilities = {token: weight / normalizer for token, weight in weights.items()}  # 计算并保存当前步骤的中间状态。
log_probs = {token: math.log(probability) for token, probability in probabilities.items()}  # 计算并保存当前步骤的中间状态。

def build_lattice(text, model_log_probs):  # 定义本节可复用的核心函数。
    edges = [[] for _ in range(len(text) + 1)]  # 计算并保存当前步骤的中间状态。
    for start in range(len(text)):  # 遍历输入元素以累积或检查结果。
        for token in sorted(model_log_probs):  # 遍历输入元素以累积或检查结果。
            if text.startswith(token, start):  # 按当前条件选择后续控制路径。
                end = start + len(token)  # 计算并保存当前步骤的中间状态。
                edges[start].append((end, token, model_log_probs[token]))  # 执行当前语句以推进本节示例。
    return edges  # 返回当前分支计算出的结果。

def show_lattice(text, edges):  # 定义本节可复用的核心函数。
    for start, outgoing in enumerate(edges[:-1]):  # 遍历输入元素以累积或检查结果。
        rendered = [(end, token, round(logp, 3)) for end, token, logp in outgoing]  # 计算并保存当前步骤的中间状态。
        print(f"位置 {start}: {rendered}")  # 执行当前语句以推进本节示例。

text = "abc"  # 计算并保存当前步骤的中间状态。
lattice = build_lattice(text, log_probs)  # 计算并保存当前步骤的中间状态。
show_lattice(text, lattice)  # 执行当前语句以推进本节示例。


## 2. Viterbi：求最佳完整路径

令 `dp[j]` 为覆盖前 `j` 个字符的最小负 log 代价：

\[
dp[j]=\min_{(i,j,t)}\{dp[i]-\log p(t)\},\quad dp[0]=0.
\]

每次更新终点时保存 `(前驱位置, token)`，最后从字符串末尾回溯。复杂度是 $O(E)$，其中 $E$ 是 lattice 边数。真实词表应用 Trie 枚举从每个起点出发的候选，而不是检查所有 token。


In [ ]:
def viterbi(text, model_log_probs):  # 定义本节可复用的核心函数。
    edges = build_lattice(text, model_log_probs)  # 计算并保存当前步骤的中间状态。
    dp = [math.inf] * (len(text) + 1)  # 计算并保存当前步骤的中间状态。
    back = [None] * (len(text) + 1)  # 计算并保存当前步骤的中间状态。
    dp[0] = 0.0  # 计算并保存当前步骤的中间状态。
    for start in range(len(text)):  # 遍历输入元素以累积或检查结果。
        if math.isinf(dp[start]):  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        for end, token, logp in edges[start]:  # 遍历输入元素以累积或检查结果。
            candidate = dp[start] - logp  # 计算并保存当前步骤的中间状态。
            if candidate < dp[end]:  # 按当前条件选择后续控制路径。
                dp[end] = candidate  # 计算并保存当前步骤的中间状态。
                back[end] = (start, token)  # 计算并保存当前步骤的中间状态。
    if back[-1] is None and text:  # 按当前条件选择后续控制路径。
        raise ValueError(f"输入 {text!r} 没有完整切分路径")  # 遇到非法合同立即显式失败。
    pieces = []  # 计算并保存当前步骤的中间状态。
    position = len(text)  # 计算并保存当前步骤的中间状态。
    while position > 0:  # 在终止条件满足前持续推进状态。
        previous, token = back[position]  # 计算并保存当前步骤的中间状态。
        pieces.append(token)  # 执行当前语句以推进本节示例。
        position = previous  # 计算并保存当前步骤的中间状态。
    return dp[-1], list(reversed(pieces))  # 返回当前分支计算出的结果。

for sample in ["abc", "unigram"]:  # 遍历输入元素以累积或检查结果。
    cost, pieces = viterbi(sample, log_probs)  # 计算并保存当前步骤的中间状态。
    print(f"{sample!r:10} -> {pieces}, cost={cost:.4f}")  # 计算并保存当前步骤的中间状态。

# abc 在词表中，但概率很低；最佳路径不一定选择最长 token。
assert viterbi("abc", log_probs)[1] != ["abc"]  # 用受控断言验证关键不变量。


## 3. Forward-backward：对所有路径求和

Viterbi 使用 `min/max` 只保留一条路径；Unigram 训练需要所有路径的边缘概率。直接连乘会下溢，因此在 log 域使用：

\[
\operatorname{LSE}(v_1,\ldots,v_k)=m+\log\sum_i e^{v_i-m},\quad m=\max_i v_i.
\]

前向值汇总到达位置的所有路径，后向值汇总从位置到终点的所有路径。边 $(i,j,t)$ 的 posterior 为：

\[
\exp(\alpha_i+\log p(t)+\beta_j-\log P(x)).
\]

把相同 token 的边 posterior 相加，就是 EM 的 E 步期望计数。


In [ ]:
NEG_INF = float("-inf")  # 计算并保存当前步骤的中间状态。

def log_add(left, right):  # 定义本节可复用的核心函数。
    if left == NEG_INF:  # 按当前条件选择后续控制路径。
        return right  # 返回当前分支计算出的结果。
    if right == NEG_INF:  # 按当前条件选择后续控制路径。
        return left  # 返回当前分支计算出的结果。
    maximum = max(left, right)  # 计算并保存当前步骤的中间状态。
    return maximum + math.log(math.exp(left - maximum) + math.exp(right - maximum))  # 返回当前分支计算出的结果。

def forward_backward(text, model_log_probs):  # 定义本节可复用的核心函数。
    edges = build_lattice(text, model_log_probs)  # 计算并保存当前步骤的中间状态。
    size = len(text) + 1  # 计算并保存当前步骤的中间状态。
    alpha = [NEG_INF] * size  # 计算并保存当前步骤的中间状态。
    alpha[0] = 0.0  # 计算并保存当前步骤的中间状态。
    for start in range(len(text)):  # 遍历输入元素以累积或检查结果。
        for end, token, logp in edges[start]:  # 遍历输入元素以累积或检查结果。
            alpha[end] = log_add(alpha[end], alpha[start] + logp)  # 计算并保存当前步骤的中间状态。

    beta = [NEG_INF] * size  # 计算并保存当前步骤的中间状态。
    beta[-1] = 0.0  # 计算并保存当前步骤的中间状态。
    for start in range(len(text) - 1, -1, -1):  # 遍历输入元素以累积或检查结果。
        for end, token, logp in edges[start]:  # 遍历输入元素以累积或检查结果。
            beta[start] = log_add(beta[start], logp + beta[end])  # 计算并保存当前步骤的中间状态。

    log_z = alpha[-1]  # 计算并保存当前步骤的中间状态。
    if log_z == NEG_INF:  # 按当前条件选择后续控制路径。
        raise ValueError(f"输入 {text!r} 没有完整切分路径")  # 遇到非法合同立即显式失败。
    expected = Counter()  # 计算并保存当前步骤的中间状态。
    for start in range(len(text)):  # 遍历输入元素以累积或检查结果。
        for end, token, logp in edges[start]:  # 遍历输入元素以累积或检查结果。
            posterior = math.exp(alpha[start] + logp + beta[end] - log_z)  # 计算并保存当前步骤的中间状态。
            expected[token] += posterior  # 计算并保存当前步骤的中间状态。
    return log_z, expected, alpha, beta  # 返回当前分支计算出的结果。

log_z, expected, alpha, beta = forward_backward("abc", log_probs)  # 计算并保存当前步骤的中间状态。
print("log P('abc') =", round(log_z, 6))  # 计算并保存当前步骤的中间状态。
print("期望 token 次数 =", {k: round(v, 4) for k, v in expected.items()})  # 计算并保存当前步骤的中间状态。
assert abs(alpha[-1] - beta[0]) < 1e-12  # 用受控断言验证关键不变量。


## 4. EM 与从大到小剪枝

固定候选词表时，EM 反复执行：

1. **E 步**：对全部句子做 forward-backward，汇总每个 token 的期望次数 $c_t$；
2. **M 步**：更新 $p(t)=c_t/\sum_u c_u$；
3. 收敛后估计删除每个候选的损失，优先删除有低损失替代路径的 token；
4. 保留 required chars、特殊 token 和 byte fallback，再在缩小词表上重新 EM。

仅按最低概率删除是错误的：一个罕见字符概率虽低，却可能是某些输入的唯一覆盖。批量剪枝太激进也可能同时删掉互为替代的 pieces。


In [ ]:
def best_cost_without(text, model_log_probs, removed_token):  # 定义本节可复用的核心函数。
    reduced = {token: score for token, score in model_log_probs.items() if token != removed_token}  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        return viterbi(text, reduced)[0]  # 返回当前分支计算出的结果。
    except ValueError:  # 捕获预期异常并验证失败分支。
        return math.inf  # 返回当前分支计算出的结果。

base_cost, _ = viterbi("abc", log_probs)  # 计算并保存当前步骤的中间状态。
for token in ["abc", "bc", "a"]:  # 遍历输入元素以累积或检查结果。
    new_cost = best_cost_without("abc", log_probs, token)  # 计算并保存当前步骤的中间状态。
    increase = new_cost - base_cost  # 计算并保存当前步骤的中间状态。
    print(f"删除 {token!r:5}: 最佳路径代价增量 = {increase}")  # 计算并保存当前步骤的中间状态。

print("注意：这是单句 Viterbi 的教学近似；生产剪枝会在全语料概率目标上估计删除损失。")  # 执行当前语句以推进本节示例。


## 5. N-best 与 subword regularization

Viterbi 返回一条最佳路径；n-best 返回分数最高的前 N 条；subword regularization 则从路径分布采样。常见温度化形式为：

\[
q_\alpha(z\mid x)=\frac{P(z)^\alpha}{\sum_{z'}P(z')^\alpha}.
\]

`alpha` 越大越集中于最佳路径，越小越平坦。训练时随机切分是一种数据增强，推理通常关闭以保证确定性和缓存命中。下面仅对短字符串穷举路径，便于验证；生产实现应在 lattice 上做 k-shortest paths 或后向采样。


In [ ]:
def enumerate_paths(text, model_log_probs):  # 定义本节可复用的核心函数。
    edges = build_lattice(text, model_log_probs)  # 计算并保存当前步骤的中间状态。
    paths = []  # 计算并保存当前步骤的中间状态。
    def visit(position, tokens, log_score):  # 定义本节可复用的核心函数。
        if position == len(text):  # 按当前条件选择后续控制路径。
            paths.append((log_score, list(tokens)))  # 执行当前语句以推进本节示例。
            return  # 返回当前分支计算出的结果。
        for end, token, logp in edges[position]:  # 遍历输入元素以累积或检查结果。
            tokens.append(token)  # 执行当前语句以推进本节示例。
            visit(end, tokens, log_score + logp)  # 执行当前语句以推进本节示例。
            tokens.pop()  # 执行当前语句以推进本节示例。
    visit(0, [], 0.0)  # 执行当前语句以推进本节示例。
    return sorted(paths, key=lambda item: (-item[0], item[1]))  # 返回当前分支计算出的结果。

paths = enumerate_paths("abc", log_probs)  # 计算并保存当前步骤的中间状态。
print("全部路径（按概率从高到低）:")  # 执行当前语句以推进本节示例。
for log_score, path in paths:  # 遍历输入元素以累积或检查结果。
    print(path, "prob=", round(math.exp(log_score), 7))  # 计算并保存当前步骤的中间状态。

def sample_from_paths(paths, alpha=1.0, seed=42):  # 定义本节可复用的核心函数。
    rng = random.Random(seed)  # 计算并保存当前步骤的中间状态。
    scaled = [alpha * score for score, _ in paths]  # 计算并保存当前步骤的中间状态。
    maximum = max(scaled)  # 计算并保存当前步骤的中间状态。
    weights = [math.exp(score - maximum) for score in scaled]  # 计算并保存当前步骤的中间状态。
    return rng.choices([path for _, path in paths], weights=weights, k=1)[0]  # 返回当前分支计算出的结果。

print("固定随机种子的采样结果:", sample_from_paths(paths, alpha=0.7, seed=7))  # 计算并保存当前步骤的中间状态。


## 6. 边界、误区与相邻算法对比

- **数值稳定**：forward/backward 用 log-sum-exp，Viterbi 才用 max/min；
- **基础覆盖**：required chars 或 byte fallback 必须保证终点可达；
- **零长度 token**：会产生自环，必须禁止；
- **剪枝后要重新 EM**：旧概率分布不再归一；
- **SentencePiece 不等于 Unigram**：它是可承载 Unigram、BPE 等算法的工具框架；
- **piece score 不是 LLM 置信度**：它只服务于表面字符串切分。

| 方法 | 训练方向 | 推理选择 | 随机切分 |
|---|---|---|---|
| Unigram | 大候选 → EM → 删除低损失项 | 全局概率 Viterbi | 天然支持 |
| BPE | 小底座 → 高频 pair merge | 固定 merge rank | 需 BPE-dropout 等扩展 |
| WordPiece | 候选收益扩词表 | 局部最长匹配 | 标准版本无 |


## 练习与面试总结

1. 用 Trie 替换 `build_lattice` 中的全词表扫描。
2. 在一个小语料上实现完整 EM，验证每轮语料似然不下降。
3. 为每个候选估计全语料删除损失，并保护 required chars。
4. 实现无需枚举全部路径的后向采样，比较不同 `alpha` 的长度分布。
5. 加入空格元符号和 byte fallback，并测试严格 round-trip。

**一分钟回答**：Unigram 给每个 piece 一个概率，把所有合法子词放进位置 lattice。确定性编码用 Viterbi 找负 log 代价最小的完整路径；训练把切分当隐变量，用 forward-backward/EM 估计期望计数，再从大候选词表中删除替代损失小的项。它比最长匹配更全局，并天然支持 n-best 与采样，但需要处理候选规模、数值稳定、基础覆盖、剪枝近似和完整模型制品兼容。
